# Nano 33 BLE Serial Sensor Reader

This notebook reads sensor data streamed from an Arduino Nano 33 BLE / Nano 33 BLE Sense over USB serial and prints it inside Jupyter.

Assumptions:

- The Arduino sketch continuously writes sensor data using `Serial.print()` / `Serial.println()`.
- The baud rate is `9600` unless you changed it in `Serial.begin(...)`.
- The Arduino Serial Monitor and PlatformIO serial monitor are closed. Only one program can own the serial port at a time.

Typical ports:

- Windows: `COM6`, `COM7`, etc.
- macOS: `/dev/cu.usbmodem...`
- Linux: `/dev/ttyACM0` or `/dev/ttyUSB0`


## 1. Install dependencies

The dependencies should have already been setup within the `tinyml` Python kernel for this notebook. 

## 2. List available serial ports

Use this to find the Nano 33 BLE port. On your Windows machine, this is likely the `COM` port that appears when the board is plugged in, for example `COM6`.

In [ ]:
from serial.tools import list_ports

ports = list(list_ports.comports())

if not ports:
    print("No serial ports found. Check the USB cable, board connection, and drivers.")
else:
    for p in ports:
        print(f"{p.device:20s} | {p.description} | {p.hwid}")


## 3. Configure the serial connection

Change `PORT` to match your board. Keep `BAUD_RATE` synchronized with the value in `Serial.begin(...)` in your Arduino code.

In [ ]:
# Change this to your actual port.
# Windows example: PORT = "COM6"
# macOS example:   PORT = "/dev/cu.usbmodem14301"
# Linux example:   PORT = "/dev/ttyACM0"

PORT = "COM6"
BAUD_RATE = 9600
READ_TIMEOUT_SECONDS = 1


## 4. Define a parser for incoming sensor lines

This parser is intentionally forgiving. It extracts numeric values from each serial line.

It handles lines such as:

```text
A: 0.12, -0.04, 0.98 | G: 1.20, 0.00, -0.30 | M: 12.1, 40.2, -5.6
0.12,-0.04,0.98,1.20,0.00,-0.30,12.1,40.2,-5.6
accel: 0.12 -0.04 0.98 gyro: 1.20 0.00 -0.30 mag: 12.1 40.2 -5.6
```

If your Arduino code prints a different format, edit `parse_sensor_line(...)`.

In [ ]:
import re
import time
from datetime import datetime

FLOAT_PATTERN = re.compile(r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?")

SENSOR_COLUMNS_9 = [
    "accel_x", "accel_y", "accel_z",
    "gyro_x", "gyro_y", "gyro_z",
    "mag_x", "mag_y", "mag_z",
]

def parse_sensor_line(line: str):
    """
    Parse one serial line from the Arduino.

    Returns a dictionary with timestamp, raw line, and parsed numeric fields.
    If fewer than 9 numbers are found, the raw line is still returned.
    """
    line = line.strip()
    numbers = [float(x) for x in FLOAT_PATTERN.findall(line)]

    row = {
        "timestamp": datetime.now().isoformat(timespec="milliseconds"),
        "raw": line,
    }

    # Common case: accel xyz, gyro xyz, magnetometer xyz
    if len(numbers) >= 9:
        for name, value in zip(SENSOR_COLUMNS_9, numbers[:9]):
            row[name] = value
    else:
        # Store whatever numbers were found as value_0, value_1, ...
        for i, value in enumerate(numbers):
            row[f"value_{i}"] = value

    return row


# Quick parser test
test_line = "A: 0.12, -0.04, 0.98 | G: 1.20, 0.00, -0.30 | M: 12.1, 40.2, -5.6"
parse_sensor_line(test_line)


## 5. Open the serial connection and print incoming lines

This cell reads from the board for a fixed amount of time and prints the incoming sensor lines directly in the notebook.

If you get a port-busy error, close PlatformIO Serial Monitor, Arduino Serial Monitor, or any other program using the same port.

In [ ]:
import serial

def print_serial_stream(port=PORT, baud_rate=BAUD_RATE, seconds=10):
    """
    Print raw serial lines from the Nano 33 BLE for `seconds` seconds.
    """
    print(f"Opening {port} at {baud_rate} baud...")

    with serial.Serial(port, baud_rate, timeout=READ_TIMEOUT_SECONDS) as ser:
        # Give the board a moment. Some boards reset when the serial port opens.
        time.sleep(2)
        ser.reset_input_buffer()

        print("Reading serial stream. Press the stop button in Jupyter to interrupt.")
        start = time.time()

        while time.time() - start < seconds:
            raw = ser.readline()
            if not raw:
                continue

            line = raw.decode("utf-8", errors="replace").strip()
            if line:
                print(line)

    print("Done.")


print_serial_stream(seconds=10)


## 6. Read, parse, and display sensor records

This version converts each line into a Python dictionary and stores the results in a Pandas DataFrame.

In [ ]:
import pandas as pd

def collect_sensor_data(port=PORT, baud_rate=BAUD_RATE, seconds=10, print_rows=True):
    """
    Collect parsed sensor data for `seconds` seconds.

    Returns:
        pandas.DataFrame
    """
    rows = []
    print(f"Opening {port} at {baud_rate} baud...")

    with serial.Serial(port, baud_rate, timeout=READ_TIMEOUT_SECONDS) as ser:
        time.sleep(2)
        ser.reset_input_buffer()

        print("Collecting data. Press the stop button in Jupyter to interrupt.")
        start = time.time()

        while time.time() - start < seconds:
            raw = ser.readline()
            if not raw:
                continue

            line = raw.decode("utf-8", errors="replace").strip()
            if not line:
                continue

            row = parse_sensor_line(line)
            rows.append(row)

            if print_rows:
                # Print compact parsed data when possible; otherwise print raw.
                if all(col in row for col in SENSOR_COLUMNS_9):
                    print(
                        f"accel=({row['accel_x']: .3f}, {row['accel_y']: .3f}, {row['accel_z']: .3f})  "
                        f"gyro=({row['gyro_x']: .3f}, {row['gyro_y']: .3f}, {row['gyro_z']: .3f})  "
                        f"mag=({row['mag_x']: .3f}, {row['mag_y']: .3f}, {row['mag_z']: .3f})"
                    )
                else:
                    print(row["raw"])

    df = pd.DataFrame(rows)
    print(f"Collected {len(df)} rows.")
    return df


df = collect_sensor_data(seconds=10, print_rows=True)
df.head()


## 7. Live notebook display

This cell updates a compact display in-place instead of printing an endless wall of text. Jupyter: now with fewer paper cuts.

In [ ]:
from IPython.display import clear_output, display

def live_sensor_display(port=PORT, baud_rate=BAUD_RATE, seconds=30, keep_last=10):
    """
    Live-updating display of the most recent parsed sensor readings.
    """
    rows = []

    with serial.Serial(port, baud_rate, timeout=READ_TIMEOUT_SECONDS) as ser:
        time.sleep(2)
        ser.reset_input_buffer()

        start = time.time()

        while time.time() - start < seconds:
            raw = ser.readline()
            if not raw:
                continue

            line = raw.decode("utf-8", errors="replace").strip()
            if not line:
                continue

            rows.append(parse_sensor_line(line))

            # Refresh the display every few rows to avoid flicker.
            if len(rows) % 3 == 0:
                clear_output(wait=True)
                elapsed = time.time() - start
                print(f"Reading from {port} at {baud_rate} baud | elapsed: {elapsed:0.1f}s | rows: {len(rows)}")
                display(pd.DataFrame(rows[-keep_last:]))

    clear_output(wait=True)
    print(f"Finished. Collected {len(rows)} rows.")
    return pd.DataFrame(rows)


live_df = live_sensor_display(seconds=30, keep_last=10)


## 8. Live line graph from the serial stream

This cell reads the Nano 33 BLE serial stream and updates a line graph inside the notebook.

It uses only `matplotlib`, `pandas`, and `pyserial`, so it works with the `tinyml` Conda environment in VS Code without requiring extra notebook widget packages.

By default, it plots the accelerometer values:

- `accel_x`
- `accel_y`
- `accel_z`

To graph gyroscope or magnetometer values instead, change the `columns` argument in the final line.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

def live_sensor_plot(port=PORT, baud_rate=BAUD_RATE, seconds=60, max_points=200, 
                     refresh_every=5, columns=("accel_x", "accel_y", "accel_z")):
    rows = []
    start = None

    print(f"Opening {port} at {baud_rate} baud...")

    with serial.Serial(port, baud_rate, timeout=READ_TIMEOUT_SECONDS) as ser:
        # Some Arduino-compatible boards reset when the serial port opens.
        time.sleep(2)
        ser.reset_input_buffer()

        start = time.time()
        print("Reading and plotting. Use the notebook stop button to interrupt.")

        while time.time() - start < seconds:
            raw = ser.readline()
            line = raw.decode("utf-8", errors="replace").strip()
            row = parse_sensor_line(line)
            row["elapsed_seconds"] = time.time() - start

            # Only plot rows that contain the requested parsed columns.
            if not all(col in row for col in columns):
                continue

            rows.append(row)

            if len(rows) % refresh_every == 0:
                plot_df = pd.DataFrame(rows[-max_points:])

                clear_output(wait=True)
                print(
                    f"Reading from {port} at {baud_rate} baud | "
                    f"elapsed: {plot_df['elapsed_seconds'].iloc[-1]:0.1f}s | "
                    f"samples: {len(rows)}"
                )

                plt.figure(figsize=(10, 5))
                for col in columns:
                    plt.plot(plot_df["elapsed_seconds"], plot_df[col], label=col)

                plt.xlabel("Elapsed time (seconds)")
                plt.ylabel("Sensor value")
                plt.title("Live Nano 33 BLE Sensor Stream")
                plt.legend(loc="upper right")
                plt.grid(True)
                plt.show()

    clear_output(wait=True)
    print(f"Finished. Collected {len(rows)} plottable rows.")

    final_df = pd.DataFrame(rows)
    if not final_df.empty:
        display(final_df.tail(10))

    return final_df


# Accelerometer live graph
plot_df = live_sensor_plot(seconds=60,columns=("accel_x", "accel_y", "accel_z"))

# Gyroscope:
# plot_df = live_sensor_plot(seconds=60, columns=("gyro_x", "gyro_y", "gyro_z"))

# Magnetometer:
# plot_df = live_sensor_plot(seconds=60, columns=("mag_x", "mag_y", "mag_z"))


## 9. Save the captured data to CSV

Useful when you later want to graph or train a simple TinyML model from captured sensor windows.

In [ ]:
output_file = "nano33ble_sensor_capture.csv"

# Save whichever dataframe you want: df or live_df
data_to_save = live_df if "live_df" in globals() else df
data_to_save.to_csv(output_file, index=False)

print(f"Saved {len(data_to_save)} rows to {output_file}")


## Troubleshooting

- **No port appears:** try a shorter USB data cable. Some USB cables are charge-only; they are tiny villains.
- **Permission denied / port busy:** close PlatformIO Serial Monitor, Arduino Serial Monitor, or another notebook using the port.
- **Gibberish output:** check that `BAUD_RATE` matches `Serial.begin(...)` in the Arduino sketch.
- **No printed setup message:** the board may reset when the serial port opens. This notebook waits 2 seconds and flushes old input before reading.
- **macOS port choice:** prefer `/dev/cu.usbmodem...` over `/dev/tty.usbmodem...` for initiating serial communication.
